# T2-bonus · Skill bundles as artefacts

## Goal

Version `skills/` properly, package it, and attach the same bundle across
two agents from one source of truth — plus a CI check that validates every
`SKILL.md`'s front matter before packaging ever runs.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../agents/contract-renewal-desk/skills/renewal-routing").exists(), "run 08 first"


## Concept

`08` built the bundle inside one agent's workspace. That's fine for one
agent; it's wrong the moment a second agent (say, a category-manager-facing
variant) wants the same negotiation-drafting skill and you're tempted to
copy-paste it. This notebook promotes `skills/` to the actual source of
truth, with the agent workspaces referencing it rather than owning private
copies — and a CI-shaped validator so a malformed `description` field fails
a PR instead of silently mis-routing in production.


## Build


In [ ]:
import shutil
from pathlib import Path

canonical = Path("../skills")
for name in ["renewal-routing", "routine-renewal", "escalation-handling", "negotiation-drafting"]:
    src = Path("../agents/contract-renewal-desk/skills") / name
    dst = canonical / name
    if src.exists() and not dst.exists():
        shutil.copytree(src, dst)
print("canonical skills/:", [p.name for p in canonical.iterdir() if p.is_dir()])


In [ ]:
def validate_skill_md(path):
    text = path.read_text()
    assert text.startswith("---"), f"{path}: missing front matter"
    front = text.split("---")[1]
    assert "name:" in front, f"{path}: missing name"
    assert "description:" in front, f"{path}: missing description"
    desc_line = next(l for l in front.splitlines() if l.strip().startswith("description:"))
    desc = desc_line.split("description:", 1)[1].strip()
    assert len(desc) >= 20, f"{path}: description too short to disambiguate selection ({len(desc)} chars)"
    return True

from pathlib import Path
failures = []
for skill_md in Path("../skills").glob("*/SKILL.md"):
    try:
        validate_skill_md(skill_md)
    except AssertionError as e:
        failures.append(str(e))

assert not failures, "\n".join(failures)
print(f"{len(list(Path('../skills').glob('*/SKILL.md')))} SKILL.md files validated")


## Verify

Same harness, same golden set, every notebook.


The validator above *is* the verification for this notebook — a CI job runs the same assertion on every PR touching `skills/`.


In [ ]:
import subprocess
# smoke-test: package the canonical skills directory as a standalone bundle
result = subprocess.run(["pac", "copilot", "pack", "--inputDirectory", "../skills", "--outputFile", "../dist/skills-bundle.zip"], capture_output=True, text=True, check=False)
print(result.returncode, result.stdout[-500:] if result.returncode == 0 else result.stderr[-500:])


## Cost


In [ ]:
print("No agent build/publish here — packaging and validation don't meter Copilot Credits.")


## Teardown


In [ ]:
print("No teardown — skills/ is now the canonical source other agents reference.")
